In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Product Launch Crew

## What We're Building

A four-agent launch planning workflow that produces one consolidated launch brief:

| Agent | Role |
|-------|------|
| PM | Define launch strategy, milestones, and decision framing |
| Marketer | Build go-to-market messaging and launch channels |
| Analyst | Design launch metrics, tracking, and KPI interpretation |
| QA | Assess launch readiness, risks, and mitigations |

## Crew Orchestration

1. PM creates strategy baseline.
2. Marketer translates strategy into campaign execution.
3. Analyst defines what success and failure look like in data.
4. QA pressure-tests readiness and risks.
5. PM synthesizes all outputs into one final launch brief.

This orchestration keeps role boundaries clear while still producing one coherent decision document.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Product Details

In [3]:
product = """
Product: AskDocs AI
Category: Developer tools — AI-powered documentation search
What it does: Lets developers ask natural language questions about their
codebase documentation and get instant, accurate answers with source links.
Integrates with GitHub, Confluence, Notion, and ReadTheDocs.

Target users: Engineering teams at mid-size companies (50-500 devs)
Pricing: $5/dev/month (Team), $12/dev/month (Enterprise)
Launch date: 3 weeks from now
Current status: Beta with 50 users, 4.2/5 satisfaction score
Key differentiator: Works with private repos (no data leaves your infra)
Competition: Mintlify, GitBook AI, ReadMe
"""

print("Product loaded: AskDocs AI")

Product loaded: AskDocs AI


## Step 3 - Create 4 Agents


In [4]:
if CREWAI_AVAILABLE:
    pm_agent = Agent(
        role="PM",
        goal="Define launch strategy and deliver the final launch brief",
        backstory=(
            "You are a product manager experienced in developer-tool launches. "
            "You align positioning, timeline, and risk-informed decisions."
        ),
        llm=llm,
        verbose=True,
    )

    marketer_agent = Agent(
        role="Marketer",
        goal="Create go-to-market messaging and channel plan for launch",
        backstory=(
            "You specialize in technical product marketing and developer audience launches. "
            "You focus on authentic messaging and channel fit."
        ),
        llm=llm,
        verbose=True,
    )

    analyst_agent = Agent(
        role="Analyst",
        goal="Define KPI framework, event tracking, and launch measurement plan",
        backstory=(
            "You are a product analytics lead who designs clear launch measurement systems "
            "that distinguish leading indicators from vanity metrics."
        ),
        llm=llm,
        verbose=True,
    )

    qa_agent = Agent(
        role="QA",
        goal="Assess launch readiness and identify operational and product risks",
        backstory=(
            "You are a launch readiness QA lead focused on failure prevention, "
            "risk severity, and mitigation planning."
        ),
        llm=llm,
        verbose=True,
    )
else:
    pm_agent = {"role": "PM"}
    marketer_agent = {"role": "Marketer"}
    analyst_agent = {"role": "Analyst"}
    qa_agent = {"role": "QA"}

print("4 agents created: PM, Marketer, Analyst, QA")


4 agents created: PM, Marketer, Analyst, QA


## Step 4 — Create Tasks

In [5]:
if CREWAI_AVAILABLE:
    pm_strategy_task = Task(
        description=f"""Create the launch strategy baseline for:

{product}

Deliver:
1. Positioning statement
2. Core value propositions
3. Launch milestones and owners
4. Decision criteria for launch readiness""",
        expected_output="PM launch strategy baseline.",
        agent=pm_agent,
    )

    marketer_task = Task(
        description="""Using the PM strategy, build a go-to-market plan:
1. Launch messaging pillars
2. Channel plan (owned, earned, community)
3. Launch-week content plan
4. Conversion-focused CTA design""",
        expected_output="Launch marketing plan and message map.",
        agent=marketer_agent,
        context=[pm_strategy_task],
    )

    analyst_task = Task(
        description="""Define launch measurement plan:
1. North star metric
2. Activation and retention KPIs
3. Event instrumentation list
4. Launch dashboard sections
5. Alert thresholds and response triggers""",
        expected_output="Analytics measurement plan for launch.",
        agent=analyst_agent,
        context=[pm_strategy_task],
    )

    qa_task = Task(
        description="""Create launch readiness and risk report:
1. Readiness checklist (product, docs, support, ops)
2. Top launch risks with likelihood and impact
3. Mitigation actions and fallback plan
4. Go/no-go recommendation criteria""",
        expected_output="QA readiness and risk report.",
        agent=qa_agent,
        context=[pm_strategy_task],
    )

    launch_brief_task = Task(
        description="""Produce one executive launch brief that merges strategy, marketing, analytics, and QA.

Structure:
1. Launch overview
2. GTM summary
3. KPI and tracking summary
4. Readiness and risk summary
5. Final go/no-go recommendation with immediate next actions

Keep it under 500 words.""",
        expected_output="Single consolidated launch brief.",
        agent=pm_agent,
        context=[pm_strategy_task, marketer_task, analyst_task, qa_task],
        markdown=True,
    )
else:
    pm_strategy_task = {"name": "PM strategy task"}
    marketer_task = {"name": "Marketing task"}
    analyst_task = {"name": "Analytics task"}
    qa_task = {"name": "QA task"}
    launch_brief_task = {"name": "Launch brief task"}

print("5 tasks created with orchestration: PM strategy -> marketer/analyst/qa -> final PM launch brief")


5 tasks created with orchestration: PM strategy -> marketer/analyst/qa -> final PM launch brief


## Step 5 — Run the Crew

In [6]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Launch brief: Proceed with guarded launch, monitor activation KPI daily, and enforce risk rollback triggers."
        self.tasks_output = [
            _DemoTaskOutput("PM Strategy", "Positioning, milestones, and launch decision criteria defined."),
            _DemoTaskOutput("Marketing", "Channel mix and launch-week messaging plan prepared."),
            _DemoTaskOutput("Analytics", "KPI framework, event plan, and alert thresholds specified."),
            _DemoTaskOutput("QA", "Readiness checklist and top risks with mitigations documented."),
            _DemoTaskOutput("Launch Brief", "One consolidated go/no-go brief drafted for leadership."),
        ]


if CREWAI_AVAILABLE:
    launch_crew = Crew(
        agents=[pm_agent, marketer_agent, analyst_agent, qa_agent],
        tasks=[pm_strategy_task, marketer_task, analyst_task, qa_task, launch_brief_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Product launch crew assembled (PM, marketer, analyst, QA).")
    result = launch_crew.kickoff()
    print("\nLaunch brief complete.")
else:
    print("Demo mode: showing crew orchestration without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing crew orchestration without live CrewAI kickoff.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("LAUNCH BRIEF")
print("=" * 60)
print(result.raw)

preview("PM Strategy", result.tasks_output[0].raw)
preview("Marketing Plan", result.tasks_output[1].raw)
preview("Analytics Plan", result.tasks_output[2].raw)
preview("QA Review", result.tasks_output[3].raw)
preview("Final Launch Brief", result.tasks_output[4].raw)


LAUNCH BRIEF
Launch brief: Proceed with guarded launch, monitor activation KPI daily, and enforce risk rollback triggers.

PM Strategy
Positioning, milestones, and launch decision criteria defined.

Marketing Plan
Channel mix and launch-week messaging plan prepared.

Analytics Plan
KPI framework, event plan, and alert thresholds specified.

QA Review
Readiness checklist and top risks with mitigations documented.

Final Launch Brief
One consolidated go/no-go brief drafted for leadership.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Four-agent crew | PM + marketer + analyst + QA |
| Orchestration pattern | PM sets baseline, specialists add depth, PM compiles final brief |
| Single final artifact | One launch brief integrates all specialist outputs |
| Educational structure | Each stage maps to a launch decision layer |

## Extensions

- Add scenario branches for delayed launch vs on-time launch
- Include budget-constrained GTM variant generation
- Add post-launch retrospective task with week-1 KPI review
- Export launch brief to markdown and PDF formats
